# 13 — v13: Optuna HPO on 3-Model Stack (XGB + LGBM + ET)

**Baseline:** v9/v10 → Kaggle RMSE **$21,755.56** | OOF RMSE **~$21,627**  
**Goal:** Replace RandomizedSearchCV with Optuna (Bayesian TPE) for smarter hyperparameter search on the proven 3-model stack.

## Why Optuna vs RandomizedSearchCV

| | RandomizedSearchCV | **Optuna (TPE)** |
|---|---|---|
| Search strategy | Random samples from fixed grid | Learns from past trials, focuses on promising regions |
| Parameter space | Discrete lists only | Continuous ranges (log-scale etc.) |
| Trial efficiency | All trials equal weight | Later trials are smarter |
| Trials needed | 40 | 100 (but finds better params) |

## What changes vs v9

| # | Change | Section |
|---|---|---|
| 1 | Install `optuna` | 1 |
| 2 | Replace RandomizedSearchCV with Optuna for XGB | 6a |
| 3 | Replace RandomizedSearchCV with Optuna for LGBM | 6b |
| 4 | Replace RandomizedSearchCV with Optuna for ET | 6c |

Everything else (features, encoding, OOF loop, Ridge meta) is identical to v9.

---
## 1. Imports & Load Data

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])
print('optuna installed.')

optuna installed.


In [2]:
import pandas as pd
import numpy as np
import warnings
import optuna
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

/Users/tehmenghai/miniconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (150634, 77)  |  Test: (16737, 76)


---
## 2. Feature Engineering

Identical to v9 — no changes.

In [3]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
CBD_LAT, CBD_LON = 1.2847, 103.8510

STREET_FREQ = train['street_name'].value_counts().to_dict()

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    df = df.copy()
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)
    df['remaining_lease']     = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']         = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate']    = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    df['street_freq']     = df['street_name'].map(STREET_FREQ).fillna(0)
    df['postal_sector']   = df['postal'].astype(str).str[:4]
    df['block_num']       = pd.to_numeric(
        df['block'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0)
    return df, new_caps

train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')

Train: (150634, 94)  |  Test: (16737, 93)


---
## 3. Per-Fold OOF Target Encoding

In [4]:
MIN_COUNT_SECTOR = 10

def oof_group_median(group_series, y_series, n_splits=5, random_state=42, min_count=1):
    groups = group_series.values
    y      = y_series.values
    encoded    = np.zeros(len(groups))
    global_med = np.median(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(groups):
        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(groups[tr_idx], y[tr_idx]):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_count}
        for i in va_idx:
            encoded[i] = fold_map.get(groups[i], global_med)
    return encoded

train_fe['town_median_price']          = oof_group_median(train_fe['town'],          train['resale_price'])
train_fe['postal_sector_median_price'] = oof_group_median(train_fe['postal_sector'], train['resale_price'],
                                                           min_count=MIN_COUNT_SECTOR)
train_fe['flat_model_median_price']    = oof_group_median(train_fe['flat_model'],    train['resale_price'])

_tmp = pd.DataFrame({
    'town':          train_fe['town'].values,
    'postal_sector': train_fe['postal_sector'].values,
    'flat_model':    train_fe['flat_model'].values,
    'resale_price':  train['resale_price'].values,
})
full_town_map   = _tmp.groupby('town')['resale_price'].median()
sector_counts   = _tmp.groupby('postal_sector')['resale_price'].count()
valid_sectors   = sector_counts[sector_counts >= MIN_COUNT_SECTOR].index
full_sector_map = _tmp[_tmp['postal_sector'].isin(valid_sectors)].groupby('postal_sector')['resale_price'].median()
full_model_map  = _tmp.groupby('flat_model')['resale_price'].median()

test_fe['town_median_price']          = test_fe['town'].map(full_town_map).fillna(full_town_map.median())
test_fe['postal_sector_median_price'] = test_fe['postal_sector'].map(full_sector_map).fillna(full_sector_map.median())
test_fe['flat_model_median_price']    = test_fe['flat_model'].map(full_model_map).fillna(full_model_map.median())

print(f'OOF encoding done. Rare sectors smoothed: {(sector_counts < MIN_COUNT_SECTOR).sum()} of {len(sector_counts)}')

OOF encoding done. Rare sectors smoothed: 32 of 598


---
## 4. Prepare X and y

In [5]:
DROP_ALL = (['id', 'resale_price', 'postal', 'block']
            + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms'])

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
assert X_test.shape[1] == X.shape[1], 'X/X_test column mismatch'

Features: 71  (num=56, cat=15)


---
## 5. Preprocessing Pipeline

In [6]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def neg_dollar_rmse(y_log_true, y_log_pred):
    return -np.sqrt(mean_squared_error(np.expm1(y_log_true), np.expm1(y_log_pred)))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

print('Preprocessor ready.')

Preprocessor ready.


---
## 6a. Optuna HPO — XGBoost

100 trials. TPE sampler focuses search on promising parameter regions after early trials.

**Continuous ranges** (vs discrete lists in RandomizedSearchCV):
- `learning_rate`: log-uniform between 0.005–0.15
- `reg_alpha`, `reg_lambda`: log-uniform across 4 orders of magnitude

In [7]:
def objective_xgb(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 500, 2000),
        'max_depth'        : trial.suggest_int('max_depth', 4, 9),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.4, 0.9),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.1, 5.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist'
    }
    pipe   = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**params))])
    scores = cross_val_score(pipe, X_train, y_train_log, cv=3,
                             scoring=dollar_rmse_scorer, n_jobs=-1)
    return -scores.mean()

study_xgb = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
print('Running XGB Optuna search (100 trials × 3-fold)...')
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

print(f'\nXGB best CV RMSE: ${study_xgb.best_value:,.0f}')
print('Best params:', study_xgb.best_params)

XGB_PARAMS = {**study_xgb.best_params,
              'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist'}

xgb_check = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
xgb_check.fit(X_train, y_train_log)
xgb_val_rmse = rmse(y_val, np.expm1(xgb_check.predict(X_val)))
print(f'XGB val RMSE: ${xgb_val_rmse:,.0f}  (v9 was ~$21,679)')

Running XGB Optuna search (100 trials × 3-fold)...


Best trial: 92. Best value: 22513.7: 100%|██████████| 100/100 [48:07<00:00, 28.87s/it]



XGB best CV RMSE: $22,514
Best params: {'n_estimators': 1658, 'max_depth': 9, 'learning_rate': 0.018295875233559838, 'subsample': 0.726189055678909, 'colsample_bytree': 0.44277074054393256, 'min_child_weight': 9, 'reg_alpha': 0.2565545823539821, 'reg_lambda': 2.496693659797454}
XGB val RMSE: $21,610  (v9 was ~$21,679)


---
## 6b. Optuna HPO — LightGBM

100 trials. `num_leaves` searched as a continuous integer up to 400.

In [9]:
def objective_lgbm(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 500, 2000),
        'max_depth'         : trial.suggest_int('max_depth', 5, 15),
        'num_leaves'        : trial.suggest_int('num_leaves', 50, 400),
        'learning_rate'     : trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample'         : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.4, 0.9),
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 60),
        'reg_alpha'         : trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda'        : trial.suggest_float('reg_lambda', 1e-8, 2.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbosity': -1
    }
    pipe   = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**params))])
    scores = cross_val_score(pipe, X_train, y_train_log, cv=3,
                             scoring=dollar_rmse_scorer, n_jobs=-1)
    return -scores.mean()

study_lgbm = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
print('Running LGBM Optuna search (100 trials × 3-fold)...')
study_lgbm.optimize(objective_lgbm, n_trials=100, show_progress_bar=True)

print(f'\nLGBM best CV RMSE: ${study_lgbm.best_value:,.0f}')
print('Best params:', study_lgbm.best_params)

LGBM_PARAMS = {**study_lgbm.best_params,
               'random_state': 42, 'n_jobs': -1, 'verbosity': -1}

lgbm_check = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
lgbm_check.fit(X_train, y_train_log)
lgbm_val_rmse = rmse(y_val, np.expm1(lgbm_check.predict(X_val)))
print(f'LGBM val RMSE: ${lgbm_val_rmse:,.0f}  (v9 was ~$21,622)')

Running LGBM Optuna search (100 trials × 3-fold)...


Best trial: 22. Best value: 22480.1:  34%|███▍      | 34/100 [58:30<1:53:33, 103.24s/it]

[W 2026-04-28 04:59:36,921] Trial 34 failed with parameters: {'n_estimators': 1205, 'max_depth': 14, 'num_leaves': 288, 'learning_rate': 0.011125816646109734, 'subsample': 0.7226205899802743, 'colsample_bytree': 0.5187639832653131, 'min_child_samples': 28, 'reg_alpha': 2.7156404934784172e-08, 'reg_lambda': 0.001406398055771584} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/tehmenghai/miniconda3/envs/ml/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/yb/1npjbkhd2mqg3r6dqz0396sh0000gn/T/ipykernel_42039/3075285350.py", line 15, in objective_lgbm
    scores = cross_val_score(pipe, X_train, y_train_log, cv=3,
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tehmenghai/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 562, in cross_val_score
  

KeyboardInterrupt: 

In [11]:
# Extract best params from completed trials                                              
print(f'Best CV RMSE from {len(study_lgbm.trials)} trials: ${study_lgbm.best_value:,.0f}')                     
print('Best params:', study_lgbm.best_params)
                                                                                                                 
LGBM_PARAMS = {**study_lgbm.best_params, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}                    
                                                                                                                 
lgbm_check = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])                        
lgbm_check.fit(X_train, y_train_log)                                                     
lgbm_val_rmse = rmse(y_val, np.expm1(lgbm_check.predict(X_val)))                         
print(f'LGBM val RMSE: ${lgbm_val_rmse:,.0f}')         

Best CV RMSE from 35 trials: $22,480
Best params: {'n_estimators': 888, 'max_depth': 14, 'num_leaves': 317, 'learning_rate': 0.027237495555808743, 'subsample': 0.9224758987157715, 'colsample_bytree': 0.45641241616804945, 'min_child_samples': 31, 'reg_alpha': 1.6454217408264245e-08, 'reg_lambda': 1.1354077850194984}
LGBM val RMSE: $21,518


---
## 6c. Optuna HPO — Extra Trees

50 trials. ET is slower per trial so fewer trials, but the continuous search space still helps.

In [12]:
def objective_et(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 200, 600),
        'max_depth'         : trial.suggest_int('max_depth', 10, 35),
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 15),
        'min_samples_leaf'  : trial.suggest_int('min_samples_leaf', 1, 6),
        'max_features'      : trial.suggest_float('max_features', 0.4, 0.9),
        'random_state': 42, 'n_jobs': -1
    }
    pipe   = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**params))])
    scores = cross_val_score(pipe, X_train, y_train_log, cv=3,
                             scoring=dollar_rmse_scorer, n_jobs=-1)
    return -scores.mean()

study_et = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
print('Running ET Optuna search (50 trials × 3-fold)...')
study_et.optimize(objective_et, n_trials=50, show_progress_bar=True)

print(f'\nET best CV RMSE: ${study_et.best_value:,.0f}')
print('Best params:', study_et.best_params)

ET_PARAMS = {**study_et.best_params, 'random_state': 42, 'n_jobs': -1}

et_check = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
et_check.fit(X_train, y_train_log)
et_val_rmse = rmse(y_val, np.expm1(et_check.predict(X_val)))
print(f'ET val RMSE: ${et_val_rmse:,.0f}  (v9 was ~$23,280)')

Running ET Optuna search (50 trials × 3-fold)...


Best trial: 17. Best value: 24348.7: 100%|██████████| 50/50 [1:18:02<00:00, 93.66s/it] 



ET best CV RMSE: $24,349
Best params: {'n_estimators': 600, 'max_depth': 24, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 0.8291283582745821}
ET val RMSE: $23,140  (v9 was ~$23,280)


---
## 7. HPO Summary

In [13]:
print('Val RMSE — Optuna vs v9 RandomizedSearchCV:')
print(f'  XGB:  ${xgb_val_rmse:>10,.0f}  (v9: ~$21,679)')
print(f'  LGBM: ${lgbm_val_rmse:>10,.0f}  (v9: ~$21,622)')
print(f'  ET:   ${et_val_rmse:>10,.0f}  (v9: ~$23,280)')

Val RMSE — Optuna vs v9 RandomizedSearchCV:
  XGB:  $    21,610  (v9: ~$21,679)
  LGBM: $    21,518  (v9: ~$21,622)
  ET:   $    23,140  (v9: ~$23,280)


---
## 8. Generate OOF Predictions (5-Fold) — 3 Models

In [14]:
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_oof  = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
et_oof   = np.zeros(len(X))

xgb_test_folds  = np.zeros((5, len(X_test)))
lgbm_test_folds = np.zeros((5, len(X_test)))
et_test_folds   = np.zeros((5, len(X_test)))

fold_rmses_xgb  = []
fold_rmses_lgbm = []
fold_rmses_et   = []

ENCODE_PAIRS = [
    ('town',          'town_median_price',          1),
    ('postal_sector', 'postal_sector_median_price', MIN_COUNT_SECTOR),
    ('flat_model',    'flat_model_median_price',    1),
]

print('Generating OOF predictions (5-fold, 3 models)...')
print(f'{"Fold":>5}  {"XGB RMSE":>12}  {"LGBM RMSE":>12}  {"ET RMSE":>12}')
print('-' * 55)

for fold, (tr_idx, va_idx) in enumerate(kf5.split(X)):
    X_tr  = X.iloc[tr_idx].copy()
    X_va  = X.iloc[va_idx].copy()
    fe_tr = train_fe.iloc[tr_idx]  # use train_fe for group cols — safe even if cols are in DROP_ALL
    fe_va = train_fe.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    y_tr_log   = np.log1p(y_tr)
    global_med_tr = np.median(y_tr)

    for group_col, price_col, min_ct in ENCODE_PAIRS:
        fold_counts = {}
        fold_vals   = {}
        for g, p in zip(fe_tr[group_col].values, y_tr):
            fold_counts[g] = fold_counts.get(g, 0) + 1
            fold_vals.setdefault(g, []).append(p)
        fold_map = {g: np.median(ps) for g, ps in fold_vals.items()
                    if fold_counts[g] >= min_ct}
        X_tr[price_col] = fe_tr[group_col].map(fold_map).fillna(global_med_tr).values
        X_va[price_col] = fe_va[group_col].map(fold_map).fillna(global_med_tr).values

    # XGBoost
    xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(**XGB_PARAMS))])
    xgb_pipe.fit(X_tr, y_tr_log)
    xgb_oof[va_idx]      = xgb_pipe.predict(X_va)
    xgb_test_folds[fold] = xgb_pipe.predict(X_test)
    fold_rmses_xgb.append(rmse(y_va, np.expm1(xgb_oof[va_idx])))

    # LightGBM
    lgbm_pipe = Pipeline([('pre', preprocessor), ('model', LGBMRegressor(**LGBM_PARAMS))])
    lgbm_pipe.fit(X_tr, y_tr_log)
    lgbm_oof[va_idx]      = lgbm_pipe.predict(X_va)
    lgbm_test_folds[fold] = lgbm_pipe.predict(X_test)
    fold_rmses_lgbm.append(rmse(y_va, np.expm1(lgbm_oof[va_idx])))

    # Extra Trees
    et_pipe = Pipeline([('pre', preprocessor), ('model', ExtraTreesRegressor(**ET_PARAMS))])
    et_pipe.fit(X_tr, y_tr_log)
    et_oof[va_idx]      = et_pipe.predict(X_va)
    et_test_folds[fold] = et_pipe.predict(X_test)
    fold_rmses_et.append(rmse(y_va, np.expm1(et_oof[va_idx])))

    print(f'{fold+1:>5}  ${fold_rmses_xgb[-1]:>10,.0f}  ${fold_rmses_lgbm[-1]:>10,.0f}  ${fold_rmses_et[-1]:>10,.0f}')

print('-' * 55)
print(f'{"Mean":>5}  ${np.mean(fold_rmses_xgb):>10,.0f}  ${np.mean(fold_rmses_lgbm):>10,.0f}  ${np.mean(fold_rmses_et):>10,.0f}')

xgb_test_avg  = xgb_test_folds.mean(axis=0)
lgbm_test_avg = lgbm_test_folds.mean(axis=0)
et_test_avg   = et_test_folds.mean(axis=0)

print('\nOOF generation complete.')

Generating OOF predictions (5-fold, 3 models)...
 Fold      XGB RMSE     LGBM RMSE       ET RMSE
-------------------------------------------------------
    1  $    21,513  $    21,448  $    23,127
    2  $    22,354  $    22,305  $    23,844
    3  $    21,656  $    21,616  $    23,371
    4  $    21,923  $    21,849  $    23,578
    5  $    21,722  $    21,668  $    23,403
-------------------------------------------------------
 Mean  $    21,834  $    21,777  $    23,465

OOF generation complete.


---
## 9. Sanity Check — Individual Models & Blends

In [15]:
print('Individual OOF RMSE:')
print(f'  XGB:  ${rmse(y, np.expm1(xgb_oof)):>10,.0f}  (v9: ~$21,890)')
print(f'  LGBM: ${rmse(y, np.expm1(lgbm_oof)):>10,.0f}  (v9: ~$21,815)')
print(f'  ET:   ${rmse(y, np.expm1(et_oof)):>10,.0f}  (v9: ~$23,594)')

blend_equal = np.expm1((xgb_oof + lgbm_oof + et_oof) / 3)
print(f'\n3-model equal-weight blend OOF RMSE: ${rmse(y, blend_equal):>8,.0f}  (v9: ~$21,752)')

Individual OOF RMSE:
  XGB:  $    21,836  (v9: ~$21,890)
  LGBM: $    21,779  (v9: ~$21,815)
  ET:   $    23,466  (v9: ~$23,594)

3-model equal-weight blend OOF RMSE: $  21,775  (v9: ~$21,752)


---
## 10. Ridge Meta-Model on 3 OOF Features

In [16]:
meta_X_train = np.column_stack([xgb_oof, lgbm_oof, et_oof])
meta_X_test  = np.column_stack([xgb_test_avg, lgbm_test_avg, et_test_avg])

print('Ridge alpha sweep (3 meta-features):')
print(f'{"alpha":>10}  {"RMSE":>12}  {"XGB coef":>10}  {"LGBM coef":>10}  {"ET coef":>10}')
print('-' * 62)

best_meta_rmse   = float('inf')
best_alpha_ridge = 1.0

for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(meta_X_train, y_log)
    meta_pred_log = ridge.predict(meta_X_train)
    meta_rmse_val = rmse(y, np.expm1(meta_pred_log))
    print(f'{alpha:>10.3f}  ${meta_rmse_val:>10,.0f}  {ridge.coef_[0]:>10.4f}  {ridge.coef_[1]:>10.4f}  {ridge.coef_[2]:>10.4f}')
    if meta_rmse_val < best_meta_rmse:
        best_meta_rmse   = meta_rmse_val
        best_alpha_ridge = alpha

print(f'\nBest Ridge alpha: {best_alpha_ridge}')
meta_model = Ridge(alpha=best_alpha_ridge)
meta_model.fit(meta_X_train, y_log)
print(f'Learned coefficients:')
print(f'  XGB:  {meta_model.coef_[0]:.4f}')
print(f'  LGBM: {meta_model.coef_[1]:.4f}')
print(f'  ET:   {meta_model.coef_[2]:.4f}')
print(f'  Intercept: {meta_model.intercept_:.4f}')
print(f'\nOOF meta-train RMSE: ${best_meta_rmse:,.0f}')
print(f'v9 OOF baseline:     ~$21,627')
print(f'Improvement:         ${21627 - best_meta_rmse:,.0f}')

Ridge alpha sweep (3 meta-features):
     alpha          RMSE    XGB coef   LGBM coef     ET coef
--------------------------------------------------------------
     0.001  $    21,654      0.3807      0.4783      0.1439
     0.010  $    21,654      0.3808      0.4782      0.1440
     0.100  $    21,654      0.3814      0.4772      0.1444
     1.000  $    21,655      0.3864      0.4686      0.1480
    10.000  $    21,660      0.3964      0.4292      0.1773
   100.000  $    21,712      0.3625      0.3665      0.2720

Best Ridge alpha: 0.001
Learned coefficients:
  XGB:  0.3807
  LGBM: 0.4783
  ET:   0.1439
  Intercept: -0.0384

OOF meta-train RMSE: $21,654
v9 OOF baseline:     ~$21,627
Improvement:         $-27


---
## 11. Generate Submission v13

In [17]:
final_log  = meta_model.predict(meta_X_test)
final_pred = np.expm1(final_log)

sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v13 = pd.DataFrame({'Id': test['id'], 'Predicted': final_pred})
sub_v13 = sub_v13.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v13_optuna.csv'
sub_v13.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {sub_v13.shape}')
print(f'Price range: ${sub_v13.Predicted.min():,.0f} – ${sub_v13.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v13.Predicted.mean():,.0f}')

Saved: ../../outputs/submissions/sub_v13_optuna.csv
Shape: (16735, 2)
Price range: $178,053 – $1,165,167
Price mean:  $450,005


---
## 12. Summary

### Score tracker

| Version | Model | OOF RMSE | Kaggle |
|---|---|---|---|
| v8 | XGB+LGBM+ET stack | ~$21,804 | $21,804.67 |
| v9/v10 | + postal-sector OOF smoothing | ~$21,627 | $21,755.56 |
| v11 | + H3 geospatial | ~$21,625 | (not submitted) |
| v12 | + CatBoost 4th model | ~$21,578 | $21,783.02 |
| **v13** | **Optuna HPO, 3-model stack** | **_(run above)_** | **_(submit if improved)_** |

### Key questions after running
- Did Optuna find lower val RMSE than RandomizedSearchCV for any model?
- Is OOF RMSE < $21,627 (v9 baseline)?
- Does the Kaggle score beat $21,755.56 (v9 best)?

### Next steps
- [ ] Pipeline refactor + MLflow (saved as pending task)
- [ ] Feature engineering (planning_area OOF, price-per-sqm interactions)